# Train model 

In [1]:
%load_ext autoreload
%autoreload 2
import torch
import torch.nn as nn
from torch.utils.data import DataLoader,Subset
from torch.optim import AdamW
from model.dataset import *
from model.model import *
import pandas as pd
import random
from sklearn.model_selection import train_test_split
plots_path = "data/movie_plots.csv"
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
MODEL_ID = "distilbert-base-uncased"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [9]:
df = pd.read_csv(plots_path)
genres = pd.unique(df['movie_category'])
num_genres = len(genres)
genres_mapping = dict()
genres_mapping_inv = dict()
for n,g in enumerate(genres):
    genres_mapping[g] = n
    genres_mapping_inv[n] = g
df['genre_id'] = df['movie_category'].map(genres_mapping)

In [10]:
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_ID)

encoder = DistilBertForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=num_genres,
    output_attentions=True,
    output_hidden_states=True
).to(device)
model = PlotClassification(encoder= encoder).to(device)


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
"""plot_dataset = PlotDataset(plots= df['movie_plot'], labels= df['genre_id'], tokenizer= tokenizer)
train_ratio = 0.8
dataset_idx = range(len(plot_dataset))
sff_dataset_idx = random.shuffle(dataset_idx)
train_idx = sff_dataset_idx[:int(len(plot_dataset)*train_ratio)]
test_idx = sff_dataset_idx[int(len(plot_dataset)*train_ratio):]
batch_size = 64
train_dataset = Subset(plot_dataset[train_idx])
test_dataset = Subset(plot_dataset[test_idx])
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)"""

In [11]:
train_df, test_df = train_test_split(df, test_size=0.1, random_state=42)

train_dataset = PlotDataset(
    plots=train_df['movie_plot'], 
    labels=train_df['genre_id'], 
    tokenizer=tokenizer
)

test_dataset = PlotDataset(
    plots=test_df['movie_plot'], 
    labels=test_df['genre_id'], 
    tokenizer=tokenizer
)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)



In [12]:
optimizer = AdamW(model.parameters(),lr = 1e-4)
criterion  = nn.CrossEntropyLoss()
n_epochs = 8
train_model(model=model,optimizer= optimizer, dataloader=train_loader,criterion= criterion, epochs = n_epochs, device= device)

epoch:8 loss: 0.5257: 100%|██████████| 76/76 [09:02<00:00,  7.13s/it]


In [13]:
test_model(model = model, dataloader=test_loader)

acc: 86.09%: 100%|██████████| 9/9 [02:00<00:00, 13.37s/it]


0.8608534322820037

In [15]:
SAVE_DIR = "./weights"
model.encoder.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)


('./weights\\tokenizer_config.json',
 './weights\\special_tokens_map.json',
 './weights\\vocab.txt',
 './weights\\added_tokens.json',
 './weights\\tokenizer.json')

In [80]:
movie_plot = (
    "In the year 2049, Officer K (Ryan Gosling), a new generation blade runner "
    "for the LAPD, discovers a long-buried secret that has the potential to plunge "
    "what's left of society into chaos. His discovery leads him on a quest to find "
    "Rick Deckard (Harrison Ford), a former blade runner who has been missing for 30 years. "
    "K must race against time to prevent a war between humans and replicants "
    "while questioning his own identity.", "he is shit"
)
tok = tokenizer(movie_plot, return_tensors="pt", padding= True, truncation= True).to(device)


In [12]:
# Check token IDs in a batch
batch = next(iter(train_loader))
print("Max token ID:", batch['input_ids'].max().item())
print("Min token ID:", batch['input_ids'].min().item())
print("Vocab size:", tokenizer.vocab_size)

# Should be: max_token_id < vocab_size

Max token ID: 50118
Min token ID: 0
Vocab size: 50265


# Test API

In [47]:
import requests

url = "http://localhost:5000/predict"
data = {
    "plot": "A tense cat-and-mouse game unfolds between a detective and a brilliant serial criminal."
}

response = requests.post(url, json=data)
print(response.json())


{'predicted_class': 2}


In [7]:
model_path = "./weights"
model = DistilBertForSequenceClassification.from_pretrained(model_path)
tokenizer = DistilBertTokenizerFast.from_pretrained(model_path)

# Annoy


In [60]:
def get_embedding(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        logits, att, hidden_states = model(**inputs)
        cls_embedding = hidden_states[-1][:, 0, :]
    return cls_embedding.squeeze().cpu().numpy()

embeddings = [get_embedding(plot) for plot in df["movie_plot"]]
embedding_dim = embeddings[0].shape[0]


In [61]:
from annoy import AnnoyIndex
ann_index = AnnoyIndex(embedding_dim, metric="angular")
for i, emb in enumerate(embeddings):
    ann_index.add_item(i, emb)
ann_index.build(10)
ann_index.save("plot_embeddings.ann")

True

In [ ]:
new_plot = "A young wizard discovers his magical powers and goes to a school of magic"

new_embedding = get_embedding(new_plot)

nearest_indices = ann_index.get_nns_by_vector(new_embedding, 5)  

nearest_plots = df.iloc[nearest_indices]["movie_plot"].tolist()

print("Nearest plots:")
for i, idx in enumerate(nearest_indices):
    print(f"{i+1}. {df.iloc[idx]['movie_plot'][:100]}...")

Nearest plots:
1. The animated movie, titled "The Whimsical Adventures of Zephyr," follows the journey of a young, mis...
2. The Pied Piper is a classic tale of a young boy who is tricked by a man named Piper into leaving his...
3. In the animated movie "Aladdin," the story unfolds in the bustling city of Agrabah, where the main c...
4. In the animated movie "Hugo: The Movie Star," we follow the journey of Hugo, a mischievous and adven...
5. In a mystical realm where magic and reality intertwine, Little Lord Fauntleroy is a young boy who po...


In [ ]:
import config
config.GENRES_MAPPING = genres_mapping
config.GENRES_MAPPING_INV = genres_mapping_inv
config.NUM_GENRES = num_genres
config.EMBEDDING_DIM = embedding_dim
with open("config.py", "w") as f:
    f.write(f"GENRES_MAPPING = {config.GENRES_MAPPING}\n")
    f.write(f"GENRES_MAPPING_INV = {config.GENRES_MAPPING_INV}\n")
    f.write(f"NUM_GENRES = {config.NUM_GENRES}\n")
    f.write(f"EMBEDDING_DIM = {config.EMBEDDING_DIM}\n")

In [6]:
import torch
import flask
from transformers import __version__ as transformers_version
from importlib.metadata import version

print(f"PyTorch: {torch.__version__}")
print(f"Flask: {flask.__version__}")
print(f"Transformers: {transformers_version}")
print(f"Annoy: {version('annoy')}")


PyTorch: 2.6.0+cu124
Flask: 3.1.2
Transformers: 4.57.3
Annoy: 1.17.3


C:\Users\anhqu\AppData\Local\Temp\ipykernel_7012\3996740435.py:7: DeprecationWarning: The '__version__' attribute is deprecated and will be removed in Flask 3.2. Use feature detection or 'importlib.metadata.version("flask")' instead.
  print(f"Flask: {flask.__version__}")
